# 🎙️ Whisper small 노인 음성 파인튜닝 (Colab)

AI Hub **명령어 음성(노인남여)** 의 키오스크 분야와 **자유대화 음성(노인남여)** 의 AI스피커 분야로 Whisper small을 추가 학습합니다.
그다음 정확도를 비교하고, 모델 서버(`/stt`)에서 바로 쓸 수 있는 형식으로 변환합니다.

> ⚠️ **AI Hub는 해외 IP(Colab 포함)에서 다운로드를 막습니다.** 그래서 데이터는 **내 컴퓨터에서** `model/scripts/prepare_aihub.py` 로 받아
> 학습용으로 줄인 뒤(16kHz FLAC zip + `manifest.csv`) Google Drive에 올리고, 이 노트북은 그 데이터로 **학습만** 합니다.

| 단계 | 하는 일 | 걸리는 시간 (A100 기준, 대략) |
|---|---|---|
| 0 | 설정 · GPU 확인 · 라이브러리 설치 · Drive 연결 | 3분 |
| 1 | Drive에 올린 데이터를 Colab 디스크로 복사 | 3~10분 |
| 2 | 데이터 확인 (시간 통계 · 예시 · 들어 보기) | 1분 |
| 3 | 원본 Whisper small 정확도 측정 (기준점) | 5~10분 |
| 4 | 파인튜닝 | 1~2시간 |
| 5 | 학습한 모델 정확도 측정 · 비교 | 5~10분 |
| 6 | faster-whisper 형식으로 변환 · 속도 확인 · Drive에 저장 | 5분 |

### 시작 전 준비
1. **데이터 올리기:** 내 컴퓨터의 `aihub_data/upload/<실험 이름>/` 폴더 안 파일 전부(zip 여러 개 + `manifest.csv`)를
   Google Drive의 `fundamental1/model/data/<실험 이름>/` 에 올립니다. (아래 `DATA_DRIVE_DIR`)
2. **GPU 켜기:** 메뉴 `런타임` → `런타임 유형 변경` → **A100 GPU** (없으면 L4) → 저장
3. **Google Drive 여유 공간:** 데이터 외에 약 2GB (결과 모델)

### 실행 방법
- 위에서부터 차례로 각 칸의 ▶ 를 누릅니다. (또는 `런타임` → `모두 실행`)
- 학습 중간 저장은 Drive 용량을 아끼려고 **Colab 디스크**에 합니다. 연결이 끊기지 않도록 Colab Pro+ 의 **백그라운드 실행**을 켜 두세요
  (`런타임` → `런타임 유형 변경` 또는 오른쪽 위 리소스 메뉴).
- A100은 컴퓨팅 단위를 빨리 씁니다. 다 끝나면 `런타임` → `런타임 연결 해제 및 삭제` 를 눌러 주세요.
- 저장은 `파일` → `Drive에 사본 저장` 으로 하고, GitHub에는 저장하지 마세요 (고칠 점은 AI에게 말하면 PR로 반영).

> 데이터 출처: AI Hub (aihub.or.kr) — 명령어 음성(노인남여), 자유대화 음성(노인남여). 데이터를 재배포하지 않습니다.

## 0. 설정
아래 값만 바꾸면 됩니다. 처음에는 그대로 실행해 보세요.

In [ ]:
# ===== 여기만 바꾸면 됩니다 =====
RUN_NAME = "whisper-small-elderly-v1"   # 실험 이름 (노션 "모델 실험 기록"에도 같은 이름으로)
DRIVE_DIR = "/content/drive/MyDrive/fundamental1/model"   # 결과를 저장할 Google Drive 폴더
DATA_DRIVE_DIR = f"{DRIVE_DIR}/data/{RUN_NAME}"            # 내 컴퓨터에서 정리해 올린 데이터 폴더

BASE_MODEL = "openai/whisper-small"
MAX_STEPS = 4000         # 학습 횟수. 빠르게 시험해 보려면 500
LEARNING_RATE = 1e-5
BATCH_SIZE = 32          # GPU 메모리가 부족하다는 에러가 나면 16 (L4는 16 권장)
EVAL_EVERY = 1000        # 몇 번 학습할 때마다 정확도를 재고 중간 저장할지
EVAL_SAMPLES = 1000      # 정확도 측정에 쓸 검증 문장 수 (데이터셋마다)
TRAIN_EVAL_SAMPLES = 200 # 학습 중간 점검에 쓸 검증 문장 수 (데이터셋마다, 적을수록 빠름)
MAX_TRAIN_SAMPLES = None # 학습 문장 수 제한 (None = 전부). 빠른 시험: 5000
SEED = 42

import os
LOCAL_DATA = f"/content/data/{RUN_NAME}"                # Colab 디스크 (빠름, 연결이 끊기면 사라짐)
CKPT_DIR = f"/content/checkpoints/{RUN_NAME}"           # 학습 중간 저장 (Drive 용량을 아끼려고 Colab 디스크)
FINAL_DIR = f"{DRIVE_DIR}/models/{RUN_NAME}"            # 학습 끝난 모델 (Hugging Face 형식)
CT2_DIR = f"{DRIVE_DIR}/models/{RUN_NAME}-ct2"          # 모델 서버용 (faster-whisper 형식)
RESULT_DIR = f"{DRIVE_DIR}/results"

In [ ]:
# GPU와 디스크 확인
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h /content | tail -1
import os
print("CPU 코어:", os.cpu_count())

In [ ]:
# 라이브러리 설치 (1~2분)
# ctranslate2 4.6.0: 4.6.3 이상은 모델 변환 때 transformers 4.46 과 맞지 않음
!pip install -q "transformers==4.46.3" "accelerate==1.1.1" "jiwer==3.0.5" "faster-whisper==1.2.1" "ctranslate2==4.6.0" "av==18.1.0"

In [ ]:
# Google Drive 연결 (처음 한 번 허용 창이 뜹니다)
from google.colab import drive
drive.mount("/content/drive")
for d in [DRIVE_DIR, RESULT_DIR, LOCAL_DATA, CKPT_DIR]:
    os.makedirs(d, exist_ok=True)
print("저장 위치:", DRIVE_DIR)

## 1. Drive에 올린 데이터 가져오기
Drive에서 바로 읽으면 느려서, 시작할 때 Colab 디스크로 한 번 복사합니다. 이미 복사했으면 건너뜁니다.

In [ ]:
import shutil, glob
import pandas as pd

if not os.path.exists(f"{DATA_DRIVE_DIR}/manifest.csv"):
    raise SystemExit(f"❌ {DATA_DRIVE_DIR}/manifest.csv 가 없습니다. 내 컴퓨터의 aihub_data/upload/{RUN_NAME}/ 안 파일을 이 Drive 폴더에 올려 주세요.")

files = sorted(os.listdir(DATA_DRIVE_DIR))
for i, f in enumerate(files, 1):
    src, dst = f"{DATA_DRIVE_DIR}/{f}", f"{LOCAL_DATA}/{f}"
    if os.path.exists(dst) and os.path.getsize(dst) == os.path.getsize(src):
        continue
    print(f"[{i}/{len(files)}] 복사 중: {f} ({os.path.getsize(src)/1e9:.2f}GB)", flush=True)
    shutil.copy(src, dst)

manifest = pd.read_csv(f"{LOCAL_DATA}/manifest.csv")
manifest["zip"] = LOCAL_DATA + "/" + manifest["zip"]
missing = sorted(set(manifest.zip) - set(glob.glob(f"{LOCAL_DATA}/*.zip")))
if missing:
    raise SystemExit(f"❌ manifest 에 있는 zip 이 없습니다 (업로드가 덜 됐을 수 있음): {[os.path.basename(m) for m in missing]}")
print(f"✅ 데이터 {len(manifest):,}문장 준비 완료")

## 2. 데이터 확인

In [ ]:
import io, re, zipfile, random
import numpy as np, soundfile as sf

_zips = {}
def load_audio(zp, member):
    key = (os.getpid(), zp)              # 학습 중 여러 프로세스가 같은 zip을 안전하게 읽도록
    if key not in _zips:
        _zips[key] = zipfile.ZipFile(zp)
    audio, sr = sf.read(io.BytesIO(_zips[key].read(member)), dtype="float32")
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    if sr != 16000:
        import librosa
        audio = librosa.resample(audio, orig_sr=sr, target_sr=16000)
    return audio

print("=== 시간 ===")
print(manifest.groupby(["dataset", "split"])["seconds"].agg(["count", "mean", "sum"])
      .assign(hours=lambda d: (d["sum"] / 3600).round(1), mean=lambda d: d["mean"].round(1)).drop(columns="sum"))
print("\n=== 예시 ===")
for _, r in manifest.sample(8, random_state=SEED).iterrows():
    print(f"[{r.dataset}/{r.split}] {r.text}")

In [ ]:
# 들어 보기: 무작위 음성 하나
from IPython.display import Audio, display
r = manifest.sample(1).iloc[0]
print(f"[{r.dataset}/{r.split}] {r.seconds}초 · 정답: {r.text}")
display(Audio(load_audio(r.zip, r.member), rate=16000))

In [ ]:
# 학습용 / 검증용 나누기 (검증용은 내 컴퓨터에서 AI Hub Validation 묶음에서 뽑아 둔 것)
train_df = manifest[manifest.split == "train"]
if MAX_TRAIN_SAMPLES:
    train_df = train_df.sample(min(MAX_TRAIN_SAMPLES, len(train_df)), random_state=SEED)
train_df = train_df.reset_index(drop=True)

def eval_subset(n):
    parts = [g.sample(min(n, len(g)), random_state=SEED) for _, g in manifest[manifest.split == "valid"].groupby("dataset")]
    return pd.concat(parts, ignore_index=True)

eval_df = eval_subset(EVAL_SAMPLES)             # 최종 정확도 비교용
train_eval_df = eval_subset(TRAIN_EVAL_SAMPLES)  # 학습 중간 점검용
print(f"학습 {len(train_df):,}개 ({train_df.seconds.sum()/3600:.1f}시간) · 검증 {len(eval_df):,}개 · 중간 점검 {len(train_eval_df):,}개")

## 3. 원본 Whisper small 정확도 측정 (기준점)
**CER(글자 오류율)** 로 잽니다. 정답 100글자 중 몇 글자를 틀렸는지를 나타내며, **낮을수록 좋습니다.**
비교를 공정하게 하려고 띄어쓰기와 문장부호는 빼고 셉니다.
> AI Hub 정답은 숫자를 한글로 적습니다("두 개"). 원본 Whisper가 "2개"로 적으면 틀린 것으로 세므로 원본 점수가 실제보다 조금 나쁘게 나올 수 있습니다.

In [ ]:
import torch, jiwer
from tqdm.auto import tqdm
from transformers import WhisperProcessor, WhisperForConditionalGeneration

processor = WhisperProcessor.from_pretrained(BASE_MODEL, language="korean", task="transcribe")

def norm(s):
    return re.sub(r"[^가-힣A-Za-z0-9]", "", s)

def cer_score(refs, preds):
    pairs = [(norm(r), norm(p)) for r, p in zip(refs, preds) if norm(r)]
    return jiwer.cer([r for r, _ in pairs], [p for _, p in pairs])

def evaluate(model, df, batch_size=32, desc="평가"):
    model.eval()
    preds = []
    rows = df.to_dict("records")
    for i in tqdm(range(0, len(rows), batch_size), desc=desc):
        chunk = rows[i:i + batch_size]
        feats = processor.feature_extractor([load_audio(r["zip"], r["member"]) for r in chunk],
                                            sampling_rate=16000, return_tensors="pt").input_features
        with torch.no_grad():
            out = model.generate(feats.to(model.device, dtype=model.dtype), language="korean",
                                 task="transcribe", max_new_tokens=128)
        preds += processor.batch_decode(out, skip_special_tokens=True)
    result = df.assign(pred=preds)
    scores = {ds: round(cer_score(g.text, g.pred), 4) for ds, g in result.groupby("dataset")}
    scores["all"] = round(cer_score(result.text, result.pred), 4)
    return scores, result

base_model = WhisperForConditionalGeneration.from_pretrained(BASE_MODEL, torch_dtype=torch.float16).to("cuda")
base_scores, base_result = evaluate(base_model, eval_df, desc="원본 모델")
print("원본 Whisper small CER:", base_scores)
for _, r in base_result.sample(5, random_state=SEED).iterrows():
    print(f"  정답: {r.text}\n  인식: {r.pred}\n")
del base_model; torch.cuda.empty_cache()

## 4. 파인튜닝
- `EVAL_EVERY` 번마다 정확도를 재고 Colab 디스크에 중간 저장합니다. 같은 세션 안에서 멈췄다가 이 칸을 다시 실행하면 이어서 합니다.
- 표의 `Cer` 값이 내려가면 잘 배우고 있는 겁니다.
- GPU 메모리 부족(`CUDA out of memory`) 에러가 나면 맨 위 `BATCH_SIZE` 를 16으로 줄이세요.

In [ ]:
from dataclasses import dataclass
from typing import Any
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer
from transformers.trainer_utils import get_last_checkpoint

class WhisperDataset(torch.utils.data.Dataset):
    def __init__(self, df):
        self.rows = df.to_dict("records")
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, i):
        r = self.rows[i]
        try:
            audio = load_audio(r["zip"], r["member"])
        except Exception:
            return self[(i + 1) % len(self)]   # 깨진 파일은 건너뜀
        return {
            "input_features": processor.feature_extractor(audio, sampling_rate=16000).input_features[0],
            "labels": processor.tokenizer(r["text"]).input_ids,
        }

@dataclass
class Collator:
    processor: Any
    decoder_start_token_id: int
    def __call__(self, features):
        batch = self.processor.feature_extractor.pad(
            [{"input_features": f["input_features"]} for f in features], return_tensors="pt")
        lab = self.processor.tokenizer.pad([{"input_ids": f["labels"]} for f in features], return_tensors="pt")
        labels = lab["input_ids"].masked_fill(lab.attention_mask.ne(1), -100)
        if (labels[:, 0] == self.decoder_start_token_id).all().item():
            labels = labels[:, 1:]
        batch["labels"] = labels
        return batch

def compute_metrics(pred):
    pad = processor.tokenizer.pad_token_id
    label_ids = np.where(pred.label_ids != -100, pred.label_ids, pad)
    pred_ids = np.where(pred.predictions != -100, pred.predictions, pad)  # 배치를 합칠 때 생긴 빈칸
    preds = processor.batch_decode(pred_ids, skip_special_tokens=True)
    refs = processor.batch_decode(label_ids, skip_special_tokens=True)
    return {"cer": cer_score(refs, preds)}

model = WhisperForConditionalGeneration.from_pretrained(BASE_MODEL)
model.generation_config.language = "korean"
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None
model.config.forced_decoder_ids = None

use_bf16 = torch.cuda.is_bf16_supported()
args = Seq2SeqTrainingArguments(
    output_dir=CKPT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    warmup_steps=min(500, MAX_STEPS // 10),
    max_steps=MAX_STEPS,
    bf16=use_bf16,
    fp16=not use_bf16,
    eval_strategy="steps",
    eval_steps=EVAL_EVERY,
    save_steps=EVAL_EVERY,
    logging_steps=50,
    predict_with_generate=True,
    generation_max_length=128,
    load_best_model_at_end=True,
    metric_for_best_model="cer",
    greater_is_better=False,
    save_total_limit=2,
    dataloader_num_workers=min(8, os.cpu_count()),
    remove_unused_columns=False,
    report_to="none",
    seed=SEED,
)
trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=WhisperDataset(train_df),
    eval_dataset=WhisperDataset(train_eval_df),
    data_collator=Collator(processor, model.config.decoder_start_token_id),
    compute_metrics=compute_metrics,
    processing_class=processor.feature_extractor,
)

last = get_last_checkpoint(CKPT_DIR) if os.path.isdir(CKPT_DIR) else None
print("이어서 학습:" if last else "처음부터 학습", last or "")
trainer.train(resume_from_checkpoint=last)

## 5. 학습한 모델 정확도 측정 · 비교

In [ ]:
from transformers import WhisperTokenizerFast
trainer.save_model(FINAL_DIR)
processor.save_pretrained(FINAL_DIR)
# 변환(6단계)에 필요한 tokenizer.json 도 함께 저장
WhisperTokenizerFast.from_pretrained(BASE_MODEL, language="korean", task="transcribe").save_pretrained(FINAL_DIR)
print("저장:", FINAL_DIR)

ft_model = trainer.model.to(torch.float16).eval()
ft_scores, ft_result = evaluate(ft_model, eval_df, desc="학습한 모델")
compare = pd.DataFrame({"원본": base_scores, "학습 후": ft_scores})
compare["개선(%p)"] = ((compare["원본"] - compare["학습 후"]) * 100).round(2)
print("CER (낮을수록 좋음)")
print(compare)

print("\n=== 달라진 예시 ===")
both = base_result[["dataset", "text", "pred"]].rename(columns={"pred": "원본"}).assign(학습후=ft_result.pred.values)
for _, r in both[both["원본"] != both["학습후"]].head(8).iterrows():
    print(f"[{r.dataset}] 정답: {r.text}\n   원본: {r['원본']}\n   학습후: {r['학습후']}\n")
both.to_csv(f"{RESULT_DIR}/{RUN_NAME}_predictions.csv", index=False)

## 6. 모델 서버용으로 변환 · 속도 확인
모델 서버(`/stt`)는 faster-whisper를 쓰므로 그 형식(CTranslate2, int8)으로 변환합니다.

In [ ]:
!ct2-transformers-converter --model "{FINAL_DIR}" --output_dir "{CT2_DIR}" --quantization int8 --copy_files tokenizer.json preprocessor_config.json --force
!ls -la "{CT2_DIR}"

In [ ]:
import time
from faster_whisper import WhisperModel

samples = eval_df.sample(5, random_state=SEED).to_dict("records")

def speed_test(device, compute_type, **kw):
    m = WhisperModel(CT2_DIR, device=device, compute_type=compute_type, **kw)
    times = []
    for r in samples:
        audio = load_audio(r["zip"], r["member"])
        t = time.perf_counter()
        segs, _ = m.transcribe(audio, language="ko", beam_size=5, vad_filter=True)
        text = " ".join(s.text.strip() for s in segs)
        times.append(time.perf_counter() - t)
        print(f"  {len(audio)/16000:4.1f}초 음성 → {times[-1]:.2f}초 | 정답: {r['text']} | 인식: {text}")
    return round(sum(times) / len(times), 2)

print("GPU:")
gpu_sec = speed_test("cuda", "float16")
print("CPU 4코어 (모델 서버와 비슷한 환경):")
cpu_sec = speed_test("cpu", "int8", cpu_threads=4)
print(f"\n평균 처리 시간  GPU {gpu_sec}초 · CPU {cpu_sec}초")

In [ ]:
# 결과 저장 + 노션 "모델 실험 기록"에 붙여 넣을 요약
import shutil, datetime, json
zip_path = shutil.make_archive(f"{DRIVE_DIR}/exports/{RUN_NAME}-ct2", "zip", CT2_DIR)

hours = manifest[manifest.split == "train"].groupby("dataset")["seconds"].sum().div(3600).round(1).to_dict()
summary = {
    "run_name": RUN_NAME, "date": str(datetime.date.today()), "base_model": BASE_MODEL,
    "train_hours_by_dataset": hours,
    "train_samples": len(train_df), "train_hours": round(train_df.seconds.sum() / 3600, 1),
    "eval_samples": len(eval_df), "max_steps": MAX_STEPS, "learning_rate": LEARNING_RATE,
    "batch_size": BATCH_SIZE, "gpu": torch.cuda.get_device_name(0),
    "cer_base": base_scores, "cer_finetuned": ft_scores,
    "seconds_per_utterance": {"gpu_fp16": gpu_sec, "cpu4_int8": cpu_sec},
    "export_zip": zip_path,
}
with open(f"{RESULT_DIR}/{RUN_NAME}.json", "w") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print(f'''### {RUN_NAME} ({summary['date']})
| 항목 | 값 |
|---|---|
| 기반 모델 | {BASE_MODEL} |
| 학습 데이터 | {hours} (시간), 총 {summary['train_samples']:,}문장 / {summary['train_hours']}시간 |
| 학습 설정 | {MAX_STEPS} steps, lr {LEARNING_RATE}, batch {BATCH_SIZE}, {summary['gpu']} |
| CER 원본 | {base_scores} |
| CER 학습 후 | {ft_scores} |
| 처리 시간(문장당) | GPU {gpu_sec}초 · CPU 4코어 {cpu_sec}초 |
| 모델 파일 | {zip_path} |
''')

## ✅ 다 끝났으면
1. 위 표를 복사해서 AI(Claude Code)에게 **"실험 결과 노션에 기록해줘"** 라고 붙여 넣으세요. 노션 "모델 실험 기록"에 정리해 줍니다.
2. Drive의 `fundamental1/model/exports/<실험 이름>-ct2.zip` 을 내 컴퓨터로 받으세요.
   - 그다음 AI에게 **"학습한 모델 연결해줘"** 라고 하면 `model/weights/` 에 넣고 `/stt` 가 새 모델을 쓰도록 바꿔 줍니다.
3. Drive 용량이 부족하면 `fundamental1/model/data/` 의 학습 데이터는 지워도 됩니다 (내 컴퓨터에 원본이 있음).
4. `런타임` → `런타임 연결 해제 및 삭제` 를 눌러 GPU 사용을 끝내 주세요.